In [1]:
import jax

In [3]:
import jax.numpy as jnp

In [4]:
jax.devices()

[CpuDevice(id=0)]

In [7]:
x = jnp.ones((3, 4, 512))
x

Array([[[1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.]],

       [[1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.]],

       [[1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.]]], dtype=float32)

In [11]:
jax.devices()[0]

CpuDevice(id=0)

In [12]:
jax.device_put(x, jax.devices()[0])

Array([[[1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.]],

       [[1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.]],

       [[1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 1., 1.]]], dtype=float32)

## Sheet 01 — Arrays, Devices & PRNG

In [ ]:
# dtype control
x_bf16 = jnp.zeros((3, 3), dtype=jnp.bfloat16)
x_f32  = x_bf16.astype(jnp.float32)
print("bfloat16:", x_bf16.dtype)
print("float32: ", x_f32.dtype)

In [ ]:
# Functional "mutation" with .at[]
x = jnp.array([1.0, 2.0, 3.0, 4.0])
x_updated = x.at[0].set(99.0)
x_added   = x.at[1:3].add(10.0)
print("original:", x)
print("after .at[0].set(99):", x_updated)
print("after .at[1:3].add(10):", x_added)

In [ ]:
# PRNG — explicit, splittable keys
key = jax.random.PRNGKey(42)
key, subkey = jax.random.split(key)
x_normal = jax.random.normal(subkey, (4, 512))
print("normal sample shape:", x_normal.shape)

# split into N keys at once
keys = jax.random.split(key, num=8)
print("8 subkeys shape:", keys.shape)

# common distributions
key, sk1, sk2, sk3 = jax.random.split(key, 4)
print("uniform:", jax.random.uniform(sk1, (2, 2)))
print("randint:", jax.random.randint(sk2, (2, 5), 0, 100))
print("bernoulli:", jax.random.bernoulli(sk3, p=0.3, shape=(6,)))

In [ ]:
# Shape ops
a = jnp.ones((2, 3, 4))
print("reshape:         ", a.reshape(2, 12).shape)
print("transpose:       ", jnp.transpose(a, (0, 2, 1)).shape)
print("swapaxes:        ", jnp.swapaxes(a, 1, 2).shape)
print("expand_dims:     ", jnp.expand_dims(a, axis=0).shape)
print("squeeze:         ", jnp.squeeze(jnp.ones((1, 3, 1)), axis=-1).shape)
b = jnp.ones((2, 3, 4))
print("concatenate -1:  ", jnp.concatenate([a, b], axis=-1).shape)
print("stack axis=0:    ", jnp.stack([a, b], axis=0).shape)

In [ ]:
# Pytrees — JAX's universal container
params = {"w": jnp.ones((4, 4)), "b": jnp.zeros(4)}

# map a fn over every leaf
scaled = jax.tree.map(lambda p: p * 0.02, params)
print("scaled w:", scaled["w"][0])

# flatten / unflatten
leaves, treedef = jax.tree.flatten(params)
print("leaves count:", len(leaves))
restored = treedef.unflatten(leaves)
print("restored keys:", list(restored.keys()))

## Sheet 02 — Core Transforms: jit · grad · vmap

In [ ]:
import functools

# jax.jit — compile & cache
params = {"w": jax.random.normal(jax.random.PRNGKey(0), (8, 8)),
          "b": jnp.zeros(8)}

@jax.jit
def forward(params, x):
    return x @ params["w"] + params["b"]

x = jax.random.normal(jax.random.PRNGKey(1), (4, 8))
out = forward(params, x)
print("forward output shape:", out.shape)

# static_argnames for non-traced Python values
@functools.partial(jax.jit, static_argnames=("training",))
def apply(params, x, training=False):
    y = x @ params["w"] + params["b"]
    if training:
        # dropout (illustrative — real impl uses key)
        return y
    return y

print("apply (eval)  shape:", apply(params, x, training=False).shape)
print("apply (train) shape:", apply(params, x, training=True).shape)

In [ ]:
# jax.grad — automatic differentiation
def loss_fn(params, x, y):
    logits = x @ params["w"] + params["b"]
    return jnp.mean((logits - y) ** 2)   # MSE for simplicity

y = jax.random.normal(jax.random.PRNGKey(2), (4, 8))

# grad of scalar loss w.r.t. params (argnums=0 default)
grads = jax.grad(loss_fn)(params, x, y)
print("grad shapes:", {k: v.shape for k, v in grads.items()})

# value AND grad in one pass
loss_val, grads = jax.value_and_grad(loss_fn)(params, x, y)
print("loss:", loss_val)

# jit-compile the backward pass too
grad_fn = jax.jit(jax.value_and_grad(loss_fn))
loss_val, grads = grad_fn(params, x, y)
print("jit loss:", loss_val)

In [ ]:
# jax.vmap — auto-batching
# write for a single example, vmap adds the batch dim
def single_forward(params, x):   # x: (D,)
    return x @ params["w"] + params["b"]

# batched version: x: (B, D) -> (B, D)
batched = jax.vmap(single_forward, in_axes=(None, 0))
x_batch = jax.random.normal(jax.random.PRNGKey(3), (6, 8))
out = batched(params, x_batch)
print("vmap batched output shape:", out.shape)

# per-sample gradients with vmap (no loops)
def single_loss(params, x, y):
    pred = x @ params["w"] + params["b"]
    return jnp.sum((pred - y) ** 2)

y_batch = jax.random.normal(jax.random.PRNGKey(4), (6, 8))
per_sample_grads = jax.vmap(
    jax.grad(single_loss), in_axes=(None, 0, 0)
)(params, x_batch, y_batch)
print("per-sample grad w shape:", per_sample_grads["w"].shape)  # (6, 8, 8)

In [ ]:
# Transform composition: jit(vmap(grad(f)))
composed = jax.jit(jax.vmap(jax.grad(single_loss), in_axes=(None, 0, 0)))
grads_fast = composed(params, x_batch, y_batch)
print("composed jit+vmap+grad w shape:", grads_fast["w"].shape)

## Sheet 03 — Transformer Building Blocks (Flax NNX)

In [ ]:
from flax import nnx

class RMSNorm(nnx.Module):
    def __init__(self, dim: int):
        self.scale = nnx.Param(jnp.ones(dim))

    def __call__(self, x):
        rms = jnp.sqrt(jnp.mean(x ** 2, axis=-1, keepdims=True) + 1e-6)
        return x / rms * self.scale.value

# quick test
rms = RMSNorm(16)
x_test = jax.random.normal(jax.random.PRNGKey(0), (2, 4, 16))
print("RMSNorm output shape:", rms(x_test).shape)

In [ ]:
class SwiGLU(nnx.Module):
    def __init__(self, d_model: int, mult: int = 4, rngs: nnx.Rngs = None):
        hidden = int(2 * d_model * mult / 3)
        self.w1 = nnx.Linear(d_model, hidden, use_bias=False, rngs=rngs)
        self.w2 = nnx.Linear(d_model, hidden, use_bias=False, rngs=rngs)
        self.w3 = nnx.Linear(hidden, d_model,  use_bias=False, rngs=rngs)

    def __call__(self, x):
        return self.w3(jax.nn.silu(self.w1(x)) * self.w2(x))

rngs = nnx.Rngs(params=jax.random.PRNGKey(0))
ffn = SwiGLU(d_model=16, rngs=rngs)
print("SwiGLU output shape:", ffn(x_test).shape)

In [ ]:
# RoPE positional embeddings
def rope_freqs(seq_len, d_head, base=10000):
    inv_freq = 1.0 / (base ** (jnp.arange(0, d_head, 2) / d_head))
    t = jnp.arange(seq_len)
    freqs = jnp.outer(t, inv_freq)                          # (T, d_head/2)
    return jnp.concatenate([freqs, freqs], axis=-1)          # (T, d_head)

def apply_rope(x, freqs):
    x1, x2 = jnp.split(x, 2, axis=-1)
    rot = jnp.concatenate([-x2, x1], axis=-1)
    return x * jnp.cos(freqs) + rot * jnp.sin(freqs)

freqs = rope_freqs(seq_len=4, d_head=16)
print("RoPE freqs shape:", freqs.shape)

# apply to a (B, H, T, d_head) query tensor
q = jax.random.normal(jax.random.PRNGKey(0), (2, 2, 4, 16))
q_rotated = apply_rope(q, freqs)
print("rotated q shape:", q_rotated.shape)

In [ ]:
class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, n_heads: int, rngs: nnx.Rngs):
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.qkv  = nnx.Linear(d_model, 3 * d_model, use_bias=False, rngs=rngs)
        self.proj = nnx.Linear(d_model, d_model,     use_bias=False, rngs=rngs)

    def __call__(self, x, mask=None):
        B, T, D = x.shape
        q, k, v = jnp.split(self.qkv(x), 3, axis=-1)   # each (B,T,D)

        def split_heads(t):
            return t.reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)

        q, k, v = map(split_heads, (q, k, v))            # (B, H, T, d_head)

        scale  = self.d_head ** -0.5
        scores = jnp.einsum("bhid,bhjd->bhij", q, k) * scale
        if mask is not None:
            scores = scores + mask
        weights = jax.nn.softmax(scores, axis=-1)
        out = jnp.einsum("bhij,bhjd->bhid", weights, v)  # (B,H,T,d_head)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, D)
        return self.proj(out)

rngs = nnx.Rngs(params=jax.random.PRNGKey(0))
mha = MultiHeadAttention(d_model=16, n_heads=2, rngs=rngs)
x_in = jax.random.normal(jax.random.PRNGKey(1), (2, 4, 16))
print("MHA output shape:", mha(x_in).shape)

In [ ]:
class TransformerBlock(nnx.Module):
    def __init__(self, d_model, n_heads, rngs):
        self.norm1 = RMSNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads, rngs)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLU(d_model, rngs=rngs)

    def __call__(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask=mask)
        x = x + self.ffn(self.norm2(x))
        return x

rngs = nnx.Rngs(params=jax.random.PRNGKey(0))
block = TransformerBlock(d_model=16, n_heads=2, rngs=rngs)
print("TransformerBlock output shape:", block(x_in).shape)

## Sheet 04 — Full GPT-Style Decoder

In [ ]:
def cross_entropy_loss(logits, targets, ignore_index=-100):
    B, T, V = logits.shape
    logits_2d  = logits.reshape(B * T, V)
    targets_1d = targets.reshape(B * T)
    log_probs  = jax.nn.log_softmax(logits_2d, axis=-1)
    nll  = -log_probs[jnp.arange(B * T), targets_1d]
    mask = targets_1d != ignore_index
    return jnp.sum(nll * mask) / jnp.sum(mask)

class GPT(nnx.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2,
                 n_heads=2, rngs: nnx.Rngs = None):
        self.tok_emb = nnx.Embed(vocab_size, d_model, rngs=rngs)
        self.blocks  = [TransformerBlock(d_model, n_heads, rngs) for _ in range(n_layers)]
        self.norm_f  = RMSNorm(d_model)
        self.head    = nnx.Linear(d_model, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, idx):
        B, T = idx.shape
        x    = self.tok_emb(idx)                          # (B, T, D)
        mask = jnp.triu(jnp.full((T, T), -1e9), k=1)[None, None]
        for block in self.blocks:
            x = block(x, mask=mask)
        x = self.norm_f(x)
        return self.head(x)                               # (B, T, vocab)

rngs  = nnx.Rngs(params=jax.random.PRNGKey(0))
model = GPT(vocab_size=256, d_model=64, n_layers=2, n_heads=2, rngs=rngs)

idx     = jax.random.randint(jax.random.PRNGKey(1), (2, 8), 0, 256)
logits  = model(idx)
print("GPT logits shape:", logits.shape)

targets = jnp.roll(idx, -1, axis=1)
loss    = cross_entropy_loss(logits, targets)
print("cross-entropy loss:", loss)

# count parameters
n_params = sum(x.size for x in jax.tree.leaves(nnx.state(model)))
print(f"{n_params / 1e3:.1f}K parameters")

## Sheet 05 — Functional Training Loop (Optax)

In [ ]:
import optax
from flax.training.train_state import TrainState

# cosine schedule with linear warmup
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=3e-4,
    warmup_steps=10,
    decay_steps=200,
    end_value=3e-5,
)

optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(schedule, b1=0.9, b2=0.95, weight_decay=0.1),
)

# split NNX model into graphdef + pure-pytree state
graphdef, params_state = nnx.split(model)

state = TrainState.create(
    apply_fn=graphdef.apply,
    params=params_state,
    tx=optimizer,
)
print("TrainState created, step:", state.step)

In [ ]:
@jax.jit
def train_step(state, batch):
    x, y = batch

    def loss_fn(params):
        logits = state.apply_fn(params, x)
        return cross_entropy_loss(logits, y)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss

# simulate a few training steps
losses = []
for step in range(5):
    x_b = jax.random.randint(jax.random.PRNGKey(step),     (4, 8), 0, 256)
    y_b = jax.random.randint(jax.random.PRNGKey(step + 100), (4, 8), 0, 256)
    state, loss = train_step(state, (x_b, y_b))
    losses.append(float(loss))
    print(f"step {step+1}: loss={loss:.4f}, lr={schedule(state.step):.2e}")

## Sheet 06 — Losses, Masking & Metrics

In [ ]:
# Label smoothing with optax
logits_demo = jax.random.normal(jax.random.PRNGKey(0), (4, 8, 256))
targets_demo = jax.random.randint(jax.random.PRNGKey(1), (4, 8), 0, 256)
V = 256

# standard cross-entropy via optax
ce_loss = optax.softmax_cross_entropy_with_integer_labels(
    logits_demo.reshape(-1, V), targets_demo.reshape(-1)
).mean()
print("cross-entropy loss:", ce_loss)

# label smoothing
smooth = 0.1
one_hot = jax.nn.one_hot(targets_demo.reshape(-1), V)
smooth_labels = one_hot * (1 - smooth) + smooth / V
smooth_loss = optax.softmax_cross_entropy(
    logits_demo.reshape(-1, V), smooth_labels
).mean()
print("label-smoothed loss:", smooth_loss)

# perplexity
ppl = jnp.exp(ce_loss)
print("perplexity:", ppl)

In [ ]:
# Causal mask
def causal_mask(T):
    mask = jnp.triu(jnp.full((T, T), -1e9), k=1)
    return mask[None, None, :, :]   # (1, 1, T, T)

print("causal mask (T=5):\n", causal_mask(5)[0, 0])

# Padding mask combined with causal mask
def make_attn_mask(pad_mask, T):
    # pad_mask: (B, T) bool, True = real token
    key_mask = jnp.where(pad_mask[:, None, None, :], 0.0, -1e9)
    return key_mask + causal_mask(T)

# example: batch of 2, seq=4, last token padded in second sequence
pad_mask = jnp.array([[True, True, True, True],
                       [True, True, True, False]])
print("combined attn mask shape:", make_attn_mask(pad_mask, T=4).shape)

## Sheet 07 — Inference & Autoregressive Generation

In [ ]:
# Greedy decode — argmax at each step
def greedy_generate(model, prompt_ids, max_new=10):
    tokens = list(prompt_ids[0])   # single sequence for simplicity
    for _ in range(max_new):
        idx_arr = jnp.array(tokens)[None, :]   # (1, T)
        logits  = model(idx_arr)               # (1, T, vocab)
        next_tok = int(jnp.argmax(logits[0, -1]))
        tokens.append(next_tok)
    return tokens

prompt = [10, 20, 30]
generated = greedy_generate(model, [prompt], max_new=5)
print("greedy generated token ids:", generated)

In [ ]:
# Sampling strategies: temperature + top-k
def top_k_sample(logits, k, key):
    # logits: (vocab,)
    topk_vals = jnp.sort(logits)[-k]              # k-th largest value
    filtered  = jnp.where(logits < topk_vals, -1e9, logits)
    return jax.random.categorical(key, filtered)

def top_p_sample(logits, p, key):
    probs       = jax.nn.softmax(logits)
    sorted_idx  = jnp.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cum_probs   = jnp.cumsum(sorted_probs)
    cutoff      = sorted_probs[jnp.searchsorted(cum_probs, p)]
    filtered    = jnp.where(probs < cutoff, -1e9, logits)
    return jax.random.categorical(key, filtered)

key = jax.random.PRNGKey(42)
sample_logits = jax.random.normal(key, (256,))

key, sk1, sk2 = jax.random.split(key, 3)
print("top-k (k=10) sample:", int(top_k_sample(sample_logits, k=10, key=sk1)))
print("top-p (p=0.9) sample:", int(top_p_sample(sample_logits, p=0.9, key=sk2)))

## Sheet 08 — Distributed Training & Sharding

In [ ]:
from jax.sharding import Mesh, PartitionSpec as P, NamedSharding
import numpy as np

# On a single CPU we only have 1 device; show the mesh API
devices = np.array(jax.devices())
print("devices:", devices)

# 1-D mesh named "data"
mesh = Mesh(devices, ("data",))
print("mesh:", mesh)

# shard a batch array along the data axis
# (on 1 device the full array lives on that device)
sharding = NamedSharding(mesh, P("data", None))
x_demo   = jnp.ones((4, 8))
x_sharded = jax.device_put(x_demo, sharding)
print("sharded array shape:", x_sharded.shape)
print("sharding:", x_sharded.sharding)

In [ ]:
# Memory levers demo: jax.checkpoint (remat)
# Recomputes activations on backward pass instead of storing them.

from functools import partial

@partial(jax.checkpoint,
         policy=jax.checkpoint_policies.nothing_saveable)
def checkpointed_block(x, block):
    return block(x)

x_ckpt = jax.random.normal(jax.random.PRNGKey(0), (2, 4, 16))
out    = checkpointed_block(x_ckpt, block)
print("checkpoint output shape:", out.shape)

# lax.scan over layers — O(1) activation stack depth
def scan_block(carry, _):
    x = carry
    x = block(x)   # reuse the same block weights (weight-tied layers)
    return x, None

x_scan, _ = jax.lax.scan(scan_block, x_ckpt, None, length=3)
print("scan output shape:", x_scan.shape)

## Sheet 09 — Debugging, Profiling & Shape Cheat Sheet

In [ ]:
# Debugging under jit — jax.debug.print (fires every call, not just tracing)
@jax.jit
def debug_forward(params, x):
    y = x @ params["w"] + params["b"]
    jax.debug.print("y mean={m}", m=y.mean())
    return y

_ = debug_forward(params, x)

In [ ]:
# Disable jit globally for eager debugging
with jax.disable_jit():
    x_eager = jax.random.normal(jax.random.PRNGKey(0), (2, 8))
    out_eager = forward(params, x_eager)
    print("eager output:", out_eager[0, :4])   # plain numpy print works

In [ ]:
# Inspect compiled XLA (StableHLO text + cost analysis)
lowered  = jax.jit(forward).lower(params, x)
compiled = lowered.compile()
cost     = compiled.cost_analysis()
print("cost analysis keys:", list(cost[0].keys()) if cost else "N/A")

# See StableHLO (first 500 chars)
hlo_text = lowered.as_text()
print(hlo_text[:500])

In [ ]:
# Shape trail summary (printed, not run as model inference)
D, H, T, B, vocab = 64, 2, 8, 2, 256
d_head = D // H

shapes = {
    "token ids":          (B, T),
    "after embedding":    (B, T, D),
    "QKV projection":     (B, T, 3 * D),
    "split into heads":   (B, H, T, d_head),
    "attention scores":   (B, H, T, T),
    "attention output":   (B, H, T, d_head),
    "FFN hidden":         (B, T, int(2 * D * 4 / 3)),
    "logits":             (B, T, vocab),
    "KV cache (per layer)": (B, H, T, d_head),
}

print(f"{'STAGE':<26}  SHAPE")
print("-" * 52)
for stage, shape in shapes.items():
    print(f"{stage:<26}  {shape}")